In [7]:
import pandas as pd
import numpy as np

In [8]:
#任务 1：基础清洗与预处理
#读取 web_security_logs.csv
data = pd.read_csv('web_security_logs.csv')
#检查是否有缺失值，如果有，请填充或删除
data.dropna(inplace=True)
#将 timestamp 列转换为真正的 Datetime 类型，并按时间先后顺序排序
data["timestamp"] = pd.to_datetime(data["timestamp"])
data.sort_values(by="timestamp", inplace=True, ascending=True)
print(data)

                      timestamp     source_ip method  status_code  \
28   2026-03-25 00:00:58.674910  192.168.1.41    GET          404   
1325 2026-03-25 00:02:58.685906      10.0.0.7    GET          200   
4788 2026-03-25 00:05:58.724909  192.168.1.35    GET          200   
1788 2026-03-25 00:07:58.688909      10.0.0.4   POST          200   
4410 2026-03-25 00:07:58.718910     10.0.0.17    GET          302   
...                         ...           ...    ...          ...   
862  2026-03-31 22:33:58.682910      10.0.0.8    GET          500   
378  2026-03-31 22:34:58.676906  192.168.1.38    GET          200   
4006 2026-03-31 22:36:58.712907  192.168.1.25    GET          200   
1598 2026-03-31 22:37:58.687910  192.168.1.26    GET          404   
1695 2026-03-31 22:37:58.687910  192.168.1.49    GET          200   

      response_time_ms           url  
28                 688        /login  
1325                94         /cart  
4788              1805        /login  
1788           

In [9]:
# 任务 2：流量特征统计
# 统计出现频率最高的 Top 10 源 IP
top_10_ips = data["source_ip"].value_counts().head(10)
print(top_10_ips)
# 计算所有请求的 平均响应时间
avg_response_time = data["response_time_ms"].mean()
print(avg_response_time)
# 统计各状态码（200, 404 等）出现的次数
status_code_counts = data["status_code"].value_counts()
print(status_code_counts)

source_ip
192.168.1.26      93
192.168.1.17      87
45.33.21.11       85
220.181.38.149    85
192.168.1.18      83
10.0.0.5          83
192.168.1.33      80
10.0.0.8          79
10.0.0.3          78
10.0.0.2          78
Name: count, dtype: int64
1011.3178
status_code
200    3564
302     462
500     254
403     242
404     241
401     237
Name: count, dtype: int64


In [10]:
# 任务 3：自动化攻击识别（核心：使用 apply）
# 编写一个函数 detect_attack(url)：
# 如果 URL 中包含 select, union, drop -> 标记为 "SQL Injection"
# 如果包含 <script> -> 标记为 "XSS"
# 如果包含 ../ -> 标记为 "Path Traversal"
# 否则标记为 "Normal"
def detect_attack(url):
    cur_url = url.lower()
    if "select" in cur_url or "union" in cur_url or "drop" in cur_url:
        return "SQL Injection"
    elif "<script>" in cur_url:
        return "XSS"
    elif "../" in cur_url:
        return "Path Traversal"
    else:
        return "Normal"
# 使用 apply() 函数生成一个新列 attack_type
data["attack_type"] = data["url"].apply(detect_attack)
print(data)

                      timestamp     source_ip method  status_code  \
28   2026-03-25 00:00:58.674910  192.168.1.41    GET          404   
1325 2026-03-25 00:02:58.685906      10.0.0.7    GET          200   
4788 2026-03-25 00:05:58.724909  192.168.1.35    GET          200   
1788 2026-03-25 00:07:58.688909      10.0.0.4   POST          200   
4410 2026-03-25 00:07:58.718910     10.0.0.17    GET          302   
...                         ...           ...    ...          ...   
862  2026-03-31 22:33:58.682910      10.0.0.8    GET          500   
378  2026-03-31 22:34:58.676906  192.168.1.38    GET          200   
4006 2026-03-31 22:36:58.712907  192.168.1.25    GET          200   
1598 2026-03-31 22:37:58.687910  192.168.1.26    GET          404   
1695 2026-03-31 22:37:58.687910  192.168.1.49    GET          200   

      response_time_ms           url attack_type  
28                 688        /login      Normal  
1325                94         /cart      Normal  
4788              

In [11]:
# 任务 4：高风险 IP 画像（核心：使用 groupby）
# 筛选出所有 attack_type 不是 "Normal" 的记录。
high_risk_data = data[data["attack_type"] != "Normal"]
# 按 source_ip 分组，统计每个 IP 的：
# 攻击总次数。
# 涉及的攻击类型种类（去重统计）。
# 该 IP 访问导致的平均响应时间。
# 将结果保存为一个名为 high_risk_ips 的新 DataFrame。
high_risk_ips = high_risk_data.groupby("source_ip").agg(
    attack_counts = ("attack_type", "count"),
    unique_attack_types = ("attack_type", "nunique"),
    avg_response_time = ("response_time_ms", "mean")
)
high_risk_ips.sort_values(by="attack_counts", inplace=True, ascending=False)
print(high_risk_ips)

              attack_counts  unique_attack_types  avg_response_time
source_ip                                                          
192.168.1.26             11                    3         786.090909
192.168.1.28             11                    3         925.181818
192.168.1.39              9                    3         908.666667
192.168.1.49              9                    3         870.333333
10.0.0.1                  8                    2         614.250000
...                     ...                  ...                ...
192.168.1.4               2                    2         856.000000
192.168.1.12              1                    1         788.000000
192.168.1.37              1                    1        1920.000000
192.168.1.46              1                    1        1641.000000
192.168.1.8               1                    1         919.000000

[71 rows x 3 columns]


In [14]:
# 任务 5：深度分析与导出
# 找出那些“既有攻击行为，又导致了 500 错误”的记录。
data = data.query("attack_type != 'Normal' and status_code == 500")
# 将最终的风险 IP 画像（任务 4 的结果）导出为 risk_report.xlsx。
data.to_excel("risk_report.xlsx")